# TriKirby Index — 2026 Season
### Emmanuel Viray and Diego Osborn — Data Analysts, UC San Diego Tritons NCAA D1 Baseball · Big West Division
### University of California, San Diego · 2026

The TriKirby Index is a methodology inspired and expanded by the [Kirby Index](https://github.com/michaelrosen3/kirby_index) that quantifies the release mechanics consistency of each UCSD pitcher's pitches relative to all NCAA Division I pitchers in the 2026 season.

## Methodology: TriKirby Index (Percentile-Based Command Metric)

### Overview
The **TriKirby Index** answers one question: *how consistent is this pitcher's release mechanics on this pitch, compared to all NCAA D1 pitchers who throw the same pitch type?*

It extends the original [Kirby Index](https://github.com/michaelrosen3/kirby_index) in two key ways:

1. **All pitch types** — The original Kirby Index evaluates fastball command only. The TriKirby Index computes a separate score for **every pitch type** a pitcher throws (Fastball, Curveball, Slider, Changeup, Cutter, Sinker, Sweeper, etc.).
2. **NCAA D1 context** — Scores are percentile-ranked against all NCAA D1 pitchers throwing the same pitch type in the 2026 season using TrackMan data.

---

## 1. Pitch-Level Data
Each pitch includes four release measurements:
- Vertical Release Angle (VRA)
- Horizontal Release Angle (HRA)
- Vertical Release Location ($vRel$)
- Horizontal Release Location ($hRel$)

---

## 2. Aggregation
For each pitcher $p$ and pitch type $t$, the **standard deviation** of each component is computed. Lower SD = more consistent mechanics.

- **NCAA baseline:** minimum **75 pitches** per pitch type
- **UCSD arsenal:** all tagged pitch types (minimum 2 pitches to compute SD)

---

## 3. NCAA-Wide Percentile Scores
Each SD is ranked within pitch type across all qualifying NCAA D1 pitchers and inverted:

$$\text{SD}_{p,t}^{\text{pct}} = 1 - \operatorname{rank}_{\text{pct}}\left(\text{SD}_{p,t}\right)$$

Higher = better command. Scores lie in $[0, 1]$.

---

## 4. Beta Weights via Random Forest
A Random Forest trained on all NCAA D1 data predicts plate location from the four release features. Feature importances become the beta weights.

---

## 5. Final Score
$$\text{TriKirby}_{p,t} = \beta_{\text{VRA}} \cdot \text{sd\_vra\_pct} + \beta_{\text{HRA}} \cdot \text{sd\_hra\_pct} + \beta_{\text{vRel}} \cdot \text{sd\_vrel\_pct} + \beta_{\text{hRel}} \cdot \text{sd\_hrel\_pct}$$

| Score | Meaning |
|---|---|
| 0.80+ | Elite release consistency vs D1 |
| 0.60–0.79 | Above average |
| 0.40–0.59 | Average |
| 0.20–0.39 | Below average |
| < 0.20 | High mechanical variability |

## 1. Imports & BigQuery Connection

In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

client = bigquery.Client(project='master-trackman-project')
print("BigQuery client ready.")

BigQuery client ready.


/opt/anaconda3/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## 2. Pull Full NCAA D1 2026 Dataset from BigQuery

In [2]:
query = """
    SELECT
        Pitcher,
        PitcherTeam,
        TaggedPitchType,
        VertRelAngle,
        HorzRelAngle,
        RelHeight,
        RelSide,
        PlateLocHeight,
        PlateLocSide
    FROM `master-trackman-project.master_trackman_dataset.2026_data`
"""

ncaa_df = client.query(query).to_dataframe()
print(f"Total NCAA D1 2026 pitches: {len(ncaa_df):,}")
print(f"Distinct teams:             {ncaa_df['PitcherTeam'].nunique()}")
display(ncaa_df.head(3))

Total NCAA D1 2026 pitches: 1,769,783
Distinct teams:             654


,Pitcher,PitcherTeam,TaggedPitchType,VertRelAngle,HorzRelAngle,RelHeight,RelSide,PlateLocHeight,PlateLocSide
0,"Ballard, Carson",GIT_YEL,Knuckleball,2.381854,-0.674236,5.06775,2.49404,3.02022,1.42834
1,"Eddy, Gavin",CAL_BEA,Other,-0.379354,-1.904042,6.44438,2.70322,2.36554,0.64923
2,"Ritchie, Austin",YOU_HAR,Other,1.070108,-0.586813,5.98803,-1.65074,1.72110,-1.58830


## 3. Clean Data & Detect Columns

In [3]:
df = ncaa_df.copy()

def pick_col(candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

PITCHER_COL   = pick_col(["Pitcher", "PitcherName", "PitcherNameFull"])
TEAM_COL      = pick_col(["PitcherTeam", "Team", "PitcherTeamAbbrev"])
PITCHTYPE_COL = pick_col(["TaggedPitchType", "PitchType", "AutoPitchType"])
VRA_COL       = pick_col(["VertRelAngle"])
HRA_COL       = pick_col(["HorzRelAngle"])
VREL_COL      = pick_col(["RelHeight", "ReleaseHeight", "RelZ"])
HREL_COL      = pick_col(["RelSide",   "ReleaseSide",   "RelX"])

print(f"Pitcher:   {PITCHER_COL}")
print(f"Team:      {TEAM_COL}")
print(f"PitchType: {PITCHTYPE_COL}")
print(f"VRA:       {VRA_COL}")
print(f"HRA:       {HRA_COL}")
print(f"vRel:      {VREL_COL}")
print(f"hRel:      {HREL_COL}")

required = [PITCHER_COL, TEAM_COL, PITCHTYPE_COL, VRA_COL, HRA_COL, VREL_COL, HREL_COL]
if any(c is None for c in required):
    raise ValueError(f"Missing columns: {[c for c in required if c is None]}")

df = df.dropna(subset=required).copy()
df[PITCHTYPE_COL] = df[PITCHTYPE_COL].astype(str).str.strip()
df[PITCHER_COL]   = df[PITCHER_COL].astype(str).str.strip()
df[TEAM_COL]      = df[TEAM_COL].astype(str).str.strip()

print(f"\nRows after cleaning: {len(df):,}")

Pitcher:   Pitcher
Team:      PitcherTeam
PitchType: TaggedPitchType
VRA:       VertRelAngle
HRA:       HorzRelAngle
vRel:      RelHeight
hRel:      RelSide

Rows after cleaning: 1,763,769


## UCSD Pitcher Roster

In [4]:
UCSD_CODE = "CSD_TRI"

ucsd_pitchers = sorted(df.loc[df[TEAM_COL] == UCSD_CODE, PITCHER_COL].unique())
print(f"UCSD pitchers found: {len(ucsd_pitchers)}")
ucsd_pitchers

UCSD pitchers found: 18


['Bowker, Austin',
 'Brown, Quincey',
 'Cazares, Julian',
 'Davidson, Garrett',
 'Elvis, Cal',
 'Etnire, Connor',
 'Gregson, Nic',
 'Henson, Kayden',
 'Huy, Nathan',
 'King, Devon',
 'Murdock, Steele',
 'Pelzman, Harry',
 'Rector, Trevor',
 'Ries, Nathan',
 'Russell, Ethan',
 'Seneres, Evan',
 'Villar, Jake',
 'Weber, Chapman']

## 4. Build NCAA-Wide Spread (75-Pitch Minimum)

Raw release SDs for every pitcher × pitch type across all NCAA D1 — this is the percentile baseline.

In [5]:
NCAA_MIN_PITCHES = 75

ncaa_spread = (
    df.groupby([PITCHER_COL, TEAM_COL, PITCHTYPE_COL])
      .agg(
          sd_vra=(VRA_COL,   lambda s: s.std(ddof=1)),
          sd_hra=(HRA_COL,   lambda s: s.std(ddof=1)),
          sd_vrel=(VREL_COL, lambda s: s.std(ddof=1)),
          sd_hrel=(HREL_COL, lambda s: s.std(ddof=1)),
          n=(VRA_COL, "size")
      )
      .reset_index()
)

ncaa_spread = ncaa_spread[
    ncaa_spread["n"] >= NCAA_MIN_PITCHES
].dropna(subset=["sd_vra", "sd_hra", "sd_vrel", "sd_hrel"]).copy()

print(f"NCAA spread entries (pitcher x pitch type): {len(ncaa_spread):,}")
print(f"Distinct pitchers:    {ncaa_spread[PITCHER_COL].nunique():,}")
print(f"Distinct pitch types: {sorted(ncaa_spread[PITCHTYPE_COL].unique())}")
ncaa_spread.head()

NCAA spread entries (pitcher x pitch type): 7,246
Distinct pitchers:    3,573
Distinct pitch types: ['ChangeUp', 'Curveball', 'Cutter', 'Fastball', 'FourSeamFastBall', 'Knuckleball', 'OneSeamFastBall', 'Sinker', 'Slider', 'Splitter', 'Sweeper', 'TwoSeamFastBall', 'Undefined']


,Pitcher,PitcherTeam,TaggedPitchType,sd_vra,sd_hra,sd_vrel,sd_hrel,n
2,"Aasland, Tristan",UCO_HUS,Fastball,0.935801,0.768354,0.128546,0.132088,99
7,"Abadie, Connor",SAN_AZT,Sinker,0.969416,0.896041,0.329985,0.255359,97
12,"Abbadessa, Jude",TUL_GRE,Fastball,0.810037,0.857694,0.157474,0.291634,188
15,"Abbadessa, Jude",TUL_GRE,Slider,1.138253,1.144477,0.142072,0.296675,214
22,"Abel, Coleson",UNI_TEX,Slider,1.010895,1.046496,0.069903,0.115208,243


## 5. NCAA-Wide Percentile Scores

Each SD is ranked within pitch type across all qualifying NCAA D1 pitchers and inverted — identical to the original Kirby Index: `1 - rank(pct=True)`.

In [6]:
PCT_COLS = {
    "sd_vra":  "sd_vra_pct",
    "sd_hra":  "sd_hra_pct",
    "sd_vrel": "sd_vrel_pct",
    "sd_hrel": "sd_hrel_pct",
}

for raw_col, pct_col in PCT_COLS.items():
    ncaa_spread[pct_col] = (
        1 - ncaa_spread.groupby(PITCHTYPE_COL)[raw_col].rank(pct=True)
    ).round(3)

print("NCAA D1 percentile score distribution:")
display(ncaa_spread[["sd_vra_pct", "sd_hra_pct", "sd_vrel_pct", "sd_hrel_pct"]].describe().round(3))

NCAA D1 percentile score distribution:


,sd_vra_pct,sd_hra_pct,sd_vrel_pct,sd_hrel_pct
count,7246.000,7246.000,7246.000,7246.000
mean,0.499,0.499,0.499,0.499
std,0.289,0.289,0.289,0.289
min,0.000,0.000,0.000,0.000
25%,0.249,0.249,0.249,0.249
50%,0.499,0.499,0.499,0.499
75%,0.749,0.749,0.749,0.749
max,1.000,1.000,1.000,1.000


## 6. Random Forest → Beta Weights (Trained on Full NCAA D1 Data)

In [7]:
FEATURES = [VRA_COL, HRA_COL, VREL_COL, HREL_COL]
TARGETS  = ["PlateLocHeight", "PlateLocSide"]

model_df = df.dropna(subset=FEATURES + TARGETS)
print(f"Training rows: {len(model_df):,}")

X = model_df[FEATURES]
y = model_df[TARGETS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

Training rows: 1,757,999


In [8]:
rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=5,
    max_features="sqrt"
)

rf.fit(X_train, y_train)

rf_importance = pd.Series(
    rf.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

beta_weights = rf_importance / rf_importance.sum()

print("RF feature importances (beta weights):")
display(beta_weights.round(4))
print(f"\nSum of betas: {beta_weights.sum():.4f}")
print(f"R\u00b2 on test set: {r2_score(y_test, rf.predict(X_test)):.4f}")

RF feature importances (beta weights):


HorzRelAngle    0.3282
VertRelAngle    0.3008
RelSide         0.2130
RelHeight       0.1579
dtype: float64


Sum of betas: 1.0000
R² on test set: 0.4120


## 7. TriKirby Index — Compute for All NCAA D1 Pitchers

In [9]:
betas = {
    "sd_vra_pct":  beta_weights[VRA_COL],
    "sd_hra_pct":  beta_weights[HRA_COL],
    "sd_vrel_pct": beta_weights[VREL_COL],
    "sd_hrel_pct": beta_weights[HREL_COL],
}

ncaa_spread["TriKirby"] = (
    betas["sd_vra_pct"]  * ncaa_spread["sd_vra_pct"]  +
    betas["sd_hra_pct"]  * ncaa_spread["sd_hra_pct"]  +
    betas["sd_vrel_pct"] * ncaa_spread["sd_vrel_pct"] +
    betas["sd_hrel_pct"] * ncaa_spread["sd_hrel_pct"]
).round(3)

print("TriKirby score distribution (all qualifying NCAA D1):")
display(ncaa_spread["TriKirby"].describe().round(3))

TriKirby score distribution (all qualifying NCAA D1):


count    7246.000
mean        0.499
std         0.166
min         0.000
25%         0.386
50%         0.505
75%         0.618
max         0.964
Name: TriKirby, dtype: float64

## 8. UCSD Spread — All Tagged Pitch Types

The NCAA baseline uses 75 pitches for a stable percentile distribution. For UCSD pitchers **all tagged pitch types are included** (minimum 2 pitches needed to compute a standard deviation). Each UCSD SD is ranked against the NCAA D1 distribution. Pitch types with no qualifying NCAA baseline will show NaN.

In [10]:
ucsd_raw = (
    df[df[TEAM_COL] == UCSD_CODE]
    .groupby([PITCHER_COL, TEAM_COL, PITCHTYPE_COL])
    .agg(
        sd_vra=(VRA_COL,   lambda s: s.std(ddof=1)),
        sd_hra=(HRA_COL,   lambda s: s.std(ddof=1)),
        sd_vrel=(VREL_COL, lambda s: s.std(ddof=1)),
        sd_hrel=(HREL_COL, lambda s: s.std(ddof=1)),
        n=(VRA_COL, "size")
    )
    .reset_index()
)

# Keep all pitch types — minimum 2 pitches to compute a standard deviation
ucsd_raw = ucsd_raw[ucsd_raw["n"] >= 5].dropna(
    subset=["sd_vra", "sd_hra", "sd_vrel", "sd_hrel"]
).copy()

print(f"UCSD pitcher-pitch type entries (all tagged): {len(ucsd_raw)}")

# Rank each UCSD SD against the NCAA D1 distribution (75-pitch baseline)
for raw_col, pct_col in PCT_COLS.items():
    def rank_vs_ncaa(row, col=raw_col):
        pt        = row[PITCHTYPE_COL]
        val       = row[col]
        ncaa_vals = ncaa_spread.loc[ncaa_spread[PITCHTYPE_COL] == pt, col]
        if len(ncaa_vals) == 0 or pd.isna(val):
            return np.nan
        return round(1 - (ncaa_vals <= val).mean(), 3)
    ucsd_raw[pct_col] = ucsd_raw.apply(rank_vs_ncaa, axis=1)

# Compute TriKirby using betas from the NCAA RF model
ucsd_raw["TriKirby"] = (
    betas["sd_vra_pct"]  * ucsd_raw["sd_vra_pct"]  +
    betas["sd_hra_pct"]  * ucsd_raw["sd_hra_pct"]  +
    betas["sd_vrel_pct"] * ucsd_raw["sd_vrel_pct"] +
    betas["sd_hrel_pct"] * ucsd_raw["sd_hrel_pct"]
).round(3)

ucsd_spread = ucsd_raw.copy()
print(f"TriKirby computed for {ucsd_spread[PITCHER_COL].nunique()} UCSD pitchers.")

UCSD pitcher-pitch type entries (all tagged): 65
TriKirby computed for 18 UCSD pitchers.


## 9. UCSD TriKirby Index — By Pitch Type

All tagged pitch types shown. Scores are percentile-ranked against all qualifying NCAA D1 pitchers (75-pitch baseline) for that pitch type. A score of 0.80 means better command than 80% of qualifying D1 pitchers. NaN means no NCAA baseline exists for that pitch type.

In [11]:
pitch_types = sorted(ucsd_spread[PITCHTYPE_COL].dropna().unique())
print("UCSD pitch types found:", pitch_types)
print(f"Total entries: {len(ucsd_spread)}")

for pt in pitch_types:
    df_pt = (
        ucsd_spread[ucsd_spread[PITCHTYPE_COL] == pt]
        .sort_values("TriKirby", ascending=False)
        .copy()
    )
    display_cols = [c for c in [PITCHER_COL, PITCHTYPE_COL, "n", "TriKirby"] if c in df_pt.columns]
    print(f"\n=== {pt} — TriKirby (vs NCAA D1) ===")
    display(df_pt[display_cols].reset_index(drop=True))

UCSD pitch types found: ['ChangeUp', 'Curveball', 'Cutter', 'Fastball', 'FourSeamFastBall', 'Sinker', 'Slider', 'Splitter', 'Sweeper']
Total entries: 65

=== ChangeUp — TriKirby (vs NCAA D1) ===


,Pitcher,TaggedPitchType,n,TriKirby
0,"Rector, Trevor",ChangeUp,46,0.626
1,"Villar, Jake",ChangeUp,127,0.618
2,"Davidson, Garrett",ChangeUp,16,0.589
3,"Ries, Nathan",ChangeUp,22,0.531
4,"Etnire, Connor",ChangeUp,12,0.494
5,"Gregson, Nic",ChangeUp,24,0.474
6,"Brown, Quincey",ChangeUp,9,0.388
7,"Murdock, Steele",ChangeUp,18,0.368
8,"Bowker, Austin",ChangeUp,17,0.237



=== Curveball — TriKirby (vs NCAA D1) ===


,Pitcher,TaggedPitchType,n,TriKirby
0,"Villar, Jake",Curveball,15,0.644
1,"Etnire, Connor",Curveball,180,0.617
2,"Gregson, Nic",Curveball,157,0.573
3,"Davidson, Garrett",Curveball,9,0.565
4,"Ries, Nathan",Curveball,92,0.269
5,"Rector, Trevor",Curveball,10,0.262
6,"Murdock, Steele",Curveball,19,0.261



=== Cutter — TriKirby (vs NCAA D1) ===


...

,Pitcher,TaggedPitchType,n,TriKirby
0,"Huy, Nathan",Slider,121,0.832
1,"Rector, Trevor",Slider,54,0.663
2,"Cazares, Julian",Slider,247,0.640
3,"Brown, Quincey",Slider,161,0.578
4,"Henson, Kayden",Slider,47,0.568
5,"King, Devon",Slider,145,0.557
6,"Etnire, Connor",Slider,105,0.548
7,"Bowker, Austin",Slider,180,0.528
8,"Villar, Jake",Slider,197,0.515
9,"Weber, Chapman",Slider,19,0.463



=== Splitter — TriKirby (vs NCAA D1) ===


,Pitcher,TaggedPitchType,n,TriKirby
0,"Murdock, Steele",Splitter,13,0.883
1,"Russell, Ethan",Splitter,12,0.873
2,"Ries, Nathan",Splitter,31,0.429
3,"Bowker, Austin",Splitter,8,0.320



=== Sweeper — TriKirby (vs NCAA D1) ===


,Pitcher,TaggedPitchType,n,TriKirby
0,"Huy, Nathan",Sweeper,16,0.881
1,"King, Devon",Sweeper,15,0.070


## 10. TriKirby Index — Full Arsenal per UCSD Pitcher

All tagged pitch types for each pitcher, sorted by TriKirby score.

## 11. Export to CSV

In [12]:
import os

export_dir = os.path.dirname(os.path.abspath("TriKirbyIndex2026.ipynb"))

component_cols = ["sd_vra_pct", "sd_hra_pct", "sd_vrel_pct", "sd_hrel_pct"]
show_cols = [PITCHTYPE_COL, "n", "TriKirby"] + component_cols

# --- Export 1: Full arsenal (one row per pitcher x pitch type, sorted by pitcher then TriKirby) ---
ucsd_arsenal = (
    ucsd_spread[[PITCHER_COL] + show_cols]
    .sort_values([PITCHER_COL, "TriKirby", "n"], ascending=[True, False, False])
    .reset_index(drop=True)
)

arsenal_path = os.path.join(export_dir, "trikirby_ucsd_arsenal_2026.csv")
ucsd_arsenal.to_csv(arsenal_path, index=False)
print(f"Arsenal exported  -> {arsenal_path}  ({len(ucsd_arsenal)} rows)")
display(ucsd_arsenal.head(10))

# --- Export 2: Pitch type rankings (all UCSD pitchers ranked within each pitch type) ---
pitch_rankings = (
    ucsd_spread[[PITCHER_COL, PITCHTYPE_COL, "n", "TriKirby"] + component_cols]
    .sort_values([PITCHTYPE_COL, "TriKirby"], ascending=[True, False])
    .reset_index(drop=True)
)

rankings_path = os.path.join(export_dir, "trikirby_ucsd_pitch_rankings_2026.csv")
pitch_rankings.to_csv(rankings_path, index=False)
print(f"Pitch rankings exported -> {rankings_path}  ({len(pitch_rankings)} rows)")
display(pitch_rankings.head(10))

Arsenal exported  -> /Users/emjviray/Desktop/TriKirby Index/TriKirby Github/trikirby_ucsd_arsenal_2026.csv  (65 rows)


,Pitcher,TaggedPitchType,n,TriKirby,sd_vra_pct,sd_hra_pct,sd_vrel_pct,sd_hrel_pct
0,"Bowker, Austin",Sinker,148,0.587,0.843,0.212,0.704,0.718
1,"Bowker, Austin",Fastball,313,0.578,0.949,0.060,0.516,0.898
2,"Bowker, Austin",Slider,180,0.528,0.784,0.079,0.540,0.851
3,"Bowker, Austin",Splitter,8,0.320,0.000,0.000,1.000,0.760
4,"Bowker, Austin",ChangeUp,17,0.237,0.013,0.000,0.595,0.655
5,"Brown, Quincey",Slider,161,0.578,0.127,0.934,0.422,0.782
6,"Brown, Quincey",Fastball,120,0.499,0.140,0.715,0.467,0.698
7,"Brown, Quincey",ChangeUp,9,0.388,0.161,0.222,0.929,0.565
8,"Cazares, Julian",Slider,247,0.640,0.457,0.772,0.810,0.569
9,"Cazares, Julian",Fastball,191,0.545,0.368,0.508,0.727,0.717


Pitch rankings exported -> /Users/emjviray/Desktop/TriKirby Index/TriKirby Github/trikirby_ucsd_pitch_rankings_2026.csv  (65 rows)


,Pitcher,TaggedPitchType,n,TriKirby,sd_vra_pct,sd_hra_pct,sd_vrel_pct,sd_hrel_pct
0,"Rector, Trevor",ChangeUp,46,0.626,0.859,0.171,0.960,0.751
1,"Villar, Jake",ChangeUp,127,0.618,0.455,0.864,0.579,0.499
2,"Davidson, Garrett",ChangeUp,16,0.589,0.052,0.965,0.918,0.524
3,"Ries, Nathan",ChangeUp,22,0.531,0.144,0.994,0.775,0.181
4,"Etnire, Connor",ChangeUp,12,0.494,0.000,0.960,0.802,0.244
5,"Gregson, Nic",ChangeUp,24,0.474,0.078,0.960,0.792,0.048
6,"Brown, Quincey",ChangeUp,9,0.388,0.161,0.222,0.929,0.565
7,"Murdock, Steele",ChangeUp,18,0.368,0.006,0.708,0.775,0.054
8,"Bowker, Austin",ChangeUp,17,0.237,0.013,0.000,0.595,0.655
9,"Villar, Jake",Curveball,15,0.644,0.164,0.967,0.677,0.799


In [13]:
component_cols = ["sd_vra_pct", "sd_hra_pct", "sd_vrel_pct", "sd_hrel_pct"]
show_cols = [PITCHTYPE_COL, "n", "TriKirby"] + component_cols

pitcher_tables = {}

for pitcher, df_p in ucsd_spread.groupby(PITCHER_COL):
    df_p  = df_p.sort_values(["TriKirby", "n"], ascending=[False, False])
    table = df_p[show_cols].reset_index(drop=True)
    table["TriKirby"] = table["TriKirby"].round(3)
    for c in component_cols:
        table[c] = table[c].round(3)
    pitcher_tables[pitcher] = table

print(f"Built {len(pitcher_tables)} pitcher tables.")
for pitcher in sorted(pitcher_tables.keys()):
    print(f"\n=== {pitcher} — Full Arsenal (UCSD 2026, vs NCAA D1) ===")
    display(pitcher_tables[pitcher])

# Combined long-form table for export
ucsd_by_pitcher_long = (
    ucsd_spread[[PITCHER_COL] + show_cols]
    .sort_values([PITCHER_COL, "TriKirby", "n"], ascending=[True, False, False])
    .reset_index(drop=True)
)

Built 18 pitcher tables.

=== Bowker, Austin — Full Arsenal (UCSD 2026, vs NCAA D1) ===


,TaggedPitchType,n,TriKirby,sd_vra_pct,sd_hra_pct,sd_vrel_pct,sd_hrel_pct
0,Sinker,148,0.587,0.843,0.212,0.704,0.718
1,Fastball,313,0.578,0.949,0.060,0.516,0.898
2,Slider,180,0.528,0.784,0.079,0.540,0.851
3,Splitter,8,0.320,0.000,0.000,1.000,0.760
4,ChangeUp,17,0.237,0.013,0.000,0.595,0.655



=== Brown, Quincey — Full Arsenal (UCSD 2026, vs NCAA D1) ===


,TaggedPitchType,n,TriKirby,sd_vra_pct,sd_hra_pct,sd_vrel_pct,sd_hrel_pct
0,Slider,161,0.578,0.127,0.934,0.422,0.782
1,Fastball,120,0.499,0.140,0.715,0.467,0.698
2,ChangeUp,9,0.388,0.161,0.222,0.929,0.565



=== Cazares, Julian — Full Arsenal (UCSD 2026, vs NCAA D1) ===


...

,TaggedPitchType,n,TriKirby,sd_vra_pct,sd_hra_pct,sd_vrel_pct,sd_hrel_pct
0,Sinker,112,0.423,0.923,0.096,0.608,0.082
1,Slider,19,0.374,0.751,0.001,0.861,0.053



=== Villar, Jake — Full Arsenal (UCSD 2026, vs NCAA D1) ===


,TaggedPitchType,n,TriKirby,sd_vra_pct,sd_hra_pct,sd_vrel_pct,sd_hrel_pct
0,Curveball,15,0.644,0.164,0.967,0.677,0.799
1,Cutter,75,0.643,0.712,0.946,0.162,0.436
2,ChangeUp,127,0.618,0.455,0.864,0.579,0.499
3,Fastball,182,0.526,0.445,0.679,0.354,0.532
4,Slider,197,0.515,0.184,0.897,0.470,0.428
5,Sinker,21,0.514,0.055,0.825,0.683,0.560



=== Weber, Chapman — Full Arsenal (UCSD 2026, vs NCAA D1) ===


,TaggedPitchType,n,TriKirby,sd_vra_pct,sd_hra_pct,sd_vrel_pct,sd_hrel_pct
0,Slider,19,0.463,0.052,0.853,0.661,0.296
1,Fastball,72,0.447,0.124,0.391,0.805,0.722
